# Dockerized Django Deployment: Nginx, PostgreSQL, Redis, Celery


## Goal

This notebook combines previous topics:

- Django/DRF
- PostgreSQL
- Redis
- Celery
- Gunicorn
- Nginx
- Docker Compose

We build a production-style multi-container deployment.


## Architecture

```text
Client Browser / API Client
           │
           ▼
        Nginx
     /static /media
           │ dynamic requests
           ▼
   Django + Gunicorn container
      │          │
      ▼          ▼
 PostgreSQL    Redis
                │
                ▼
       Celery worker / beat
```

Services:

| Service | Responsibility |
|---------|----------------|
| `nginx` | Public reverse proxy, static/media serving |
| `web` | Django/DRF app running with Gunicorn |
| `db` | PostgreSQL database |
| `redis` | Cache and Celery broker |
| `celery_worker` | Executes background tasks |
| `celery_beat` | Runs scheduled tasks |


## Production Dockerfile for Django

```dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

RUN apt-get update \
    && apt-get install -y --no-install-recommends build-essential libpq-dev \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt /app/
RUN pip install --no-cache-dir -r requirements.txt

COPY . /app/
COPY entrypoint.sh /entrypoint.sh
RUN chmod +x /entrypoint.sh

ENTRYPOINT ["/entrypoint.sh"]
CMD ["gunicorn", "project_name.wsgi:application", "--bind", "0.0.0.0:8000", "--workers", "3"]
```

Replace `project_name` with the Django package that contains `wsgi.py`.

The image still ends with Gunicorn as the default command. The `ENTRYPOINT` script is a small startup wrapper that can wait for PostgreSQL, optionally run migrations, optionally collect static files, and then hand control to the final command.



## Entrypoint Script

Create `entrypoint.sh` next to the `Dockerfile`:

```sh
#!/bin/sh
set -e

if [ -n "$WAIT_FOR_DB_HOST" ]; then
    DB_PORT="${WAIT_FOR_DB_PORT:-5432}"
    echo "Waiting for database at $WAIT_FOR_DB_HOST:$DB_PORT..."
    python - <<'PY'
import os
import socket
import time

host = os.environ["WAIT_FOR_DB_HOST"]
port = int(os.environ.get("WAIT_FOR_DB_PORT", "5432"))

while True:
    try:
        with socket.create_connection((host, port), timeout=2):
            break
    except OSError:
        print("Database is not ready yet; waiting...")
        time.sleep(1)
PY
fi

if [ "$RUN_MIGRATIONS" = "1" ]; then
    echo "Running database migrations..."
    python manage.py migrate
fi

if [ "$COLLECT_STATIC" = "1" ]; then
    echo "Collecting static files..."
    python manage.py collectstatic --noinput
fi

exec "$@"
```

Make it executable:

```bash
chmod +x entrypoint.sh
```

Important idea:

```text
Dockerfile builds the image.
entrypoint.sh runs when a container starts.
```

So do **not** put `python manage.py migrate` as a `RUN` command inside the `Dockerfile`. Migration needs the real running database, so it belongs to container startup or, even better in production, an explicit deployment step.

The final line is important:

```sh
exec "$@"
```

It means: after the setup work finishes, run the command passed by Docker Compose. For the `web` service, that command is Gunicorn. For Celery services, it can be the Celery worker or beat command.



## `.dockerignore`

```dockerignore
.git/
__pycache__/
*.pyc
venv/
.env
.env.*
!.env.example
db.sqlite3
static_collected/
media/
.pytest_cache/
.mypy_cache/
```

Never bake real `.env` secrets into an image.


## Production Compose File

```yaml
services:
  web:
    build: .
    command: gunicorn project_name.wsgi:application --bind 0.0.0.0:8000 --workers 3
    env_file:
      - .env
    environment:
      WAIT_FOR_DB_HOST: db
      WAIT_FOR_DB_PORT: 5432
      RUN_MIGRATIONS: "1"
      COLLECT_STATIC: "1"
    depends_on:
      - db
      - redis
    volumes:
      - static_data:/app/static_collected
      - media_data:/app/media
    restart: unless-stopped

  db:
    image: postgres:15
    env_file:
      - .env
    volumes:
      - pgdata:/var/lib/postgresql/data
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    restart: unless-stopped

  celery_worker:
    build: .
    command: celery -A project_name worker -l info
    env_file:
      - .env
    environment:
      WAIT_FOR_DB_HOST: db
      WAIT_FOR_DB_PORT: 5432
    depends_on:
      - redis
      - db
    restart: unless-stopped

  celery_beat:
    build: .
    command: celery -A project_name beat -l info
    env_file:
      - .env
    environment:
      WAIT_FOR_DB_HOST: db
      WAIT_FOR_DB_PORT: 5432
    depends_on:
      - redis
      - db
    restart: unless-stopped

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/conf.d/default.conf:ro
      - static_data:/static:ro
      - media_data:/media:ro
    depends_on:
      - web
    restart: unless-stopped

volumes:
  pgdata:
  static_data:
  media_data:
```

This is a template. Real projects may need changes.

`depends_on` controls startup order, but it does not guarantee PostgreSQL is ready to accept connections. That is why the entrypoint script waits for `db:5432` before starting Django/Celery commands.

For serious production with multiple `web` replicas, do not let every replica run migrations. Prefer a one-off command or CI/CD deployment step. In this beginner template, `RUN_MIGRATIONS: "1"` is shown on `web` so the behavior is visible and easy to understand.



## Environment Variables

`.env` example:

```dotenv
DJANGO_DEBUG=False
DJANGO_SECRET_KEY=<strong-secret>
DJANGO_ALLOWED_HOSTS=SERVER_IP,api.example.com
DJANGO_CSRF_TRUSTED_ORIGINS=https://api.example.com
DATABASE_URL=postgres://user:password@db:5432/mydatabase
POSTGRES_USER=user
POSTGRES_PASSWORD=password
POSTGRES_DB=mydatabase
CELERY_BROKER_URL=redis://redis:6379/0
CELERY_RESULT_BACKEND=redis://redis:6379/1
```

Best practices:

- Do not commit real `.env`.
- Commit `.env.example` with placeholders.
- Use different secrets for dev/staging/production.
- Rotate secrets if they leak.


## Nginx Config for Docker

`nginx.conf`:

```nginx
server {
    listen 80;
    server_name SERVER_IP api.example.com;

    client_max_body_size 20m;

    location /static/ {
        alias /static/;
    }

    location /media/ {
        alias /media/;
    }

    location / {
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        proxy_pass http://web:8000;
    }
}
```

Important part:

```nginx
proxy_pass http://web:8000;
```

`web` is the Compose service name for the Django/Gunicorn container.


## Static and Media Volumes

Production static flow:

```text
web container runs collectstatic
        │
        ▼
static_data volume
        │
        ▼
nginx serves /static/
```

Run collectstatic:

```bash
docker compose exec web python manage.py collectstatic --noinput
```

Media files:

```text
Django writes uploaded files → media_data volume → Nginx serves /media/
```

For larger production systems, media files are often stored in object storage such as S3 or compatible services.


## Celery with Redis

Celery worker service:

```yaml
celery_worker:
  build: .
  command: celery -A project_name worker -l info
  env_file:
    - .env
  depends_on:
    - redis
    - db
```

Celery beat service:

```yaml
celery_beat:
  build: .
  command: celery -A project_name beat -l info
  env_file:
    - .env
  depends_on:
    - redis
    - db
```

Redis URL:

```dotenv
CELERY_BROKER_URL=redis://redis:6379/0
```

Again, `redis` is the Compose service name.


## Running on a Server

On the VPS, install Docker and copy the project:

```bash
rsync -av --delete \
  --exclude '.git' \
  --exclude '.env' \
  --exclude '.env.*' \
  --exclude '__pycache__' \
  ./ web@SERVER_IP:/srv/api/
```

SSH to server:

```bash
ssh web@SERVER_IP
cd /srv/api
```

Create production `.env`, make sure `entrypoint.sh` is executable, then run:

```bash
chmod +x entrypoint.sh
docker compose up --build -d
```

With the example environment flags above, the `web` container waits for PostgreSQL, runs migrations, collects static files, and then starts Gunicorn.

A more controlled production deployment can keep these steps manual instead:

```bash
docker compose run --rm web python manage.py migrate
docker compose run --rm web python manage.py collectstatic --noinput
docker compose up -d
```


## Logs and Operations

Show services:

```bash
docker compose ps
```

Follow all logs:

```bash
docker compose logs -f
```

Follow one service:

```bash
docker compose logs -f web
```

Restart one service:

```bash
docker compose restart web
```

Rebuild after dependency changes:

```bash
docker compose up --build -d
```

Stop services:

```bash
docker compose down
```


## Optional: Docker Hub

For a first deployment, building the image directly on the server is simpler:

```bash
docker compose up --build -d
```

Later, teams often build images in CI/CD and push them to Docker Hub or another registry:

```bash
docker build -t username/my-django-app:latest .
docker push username/my-django-app:latest
```

This is optional for this course.


## Restart Policy and Simple Scaling Note

A useful production option is:

```yaml
restart: unless-stopped
```

This tells Docker to restart containers after crashes or server reboot unless you manually stopped them.

Compose can also scale services:

```bash
docker compose up --scale web=3 -d
```

But real scaling needs load balancing and more production design. Treat scaling as optional curiosity for now.


## HTTPS

For production HTTPS, common options:

1. Install Certbot on the host and configure host Nginx.
2. Run a separate reverse proxy container that handles certificates.
3. Use a cloud load balancer or CDN in front of the server.

For learning, it is okay to first deploy HTTP, then add HTTPS after the app works.

Do not ignore HTTPS for real production.


## Common Issues

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Django cannot connect to DB | Using `localhost` instead of `db` | Use service name `db` in `DATABASE_URL` |
| Static files 404 | `collectstatic` not run or volume mismatch | Run collectstatic; check Nginx static volume |
| Entrypoint permission denied | Script is not executable or copied incorrectly | Run `chmod +x entrypoint.sh`; check Dockerfile `COPY` path |
| Celery cannot connect to Redis | Wrong Redis URL | Use `redis://redis:6379/0` |
| Nginx 502 | Web container not running or wrong `proxy_pass` | Check `docker compose ps` and logs |
| Data lost | Used `down -v` or no volume | Use named volumes and avoid deleting them |
| Env changes not applied | Container not recreated/restarted | Restart or recreate service |


## Mini Project

Deploy a full Django application with Docker Compose:

1. Django/DRF app running with Gunicorn.
2. PostgreSQL service with persistent volume.
3. Redis service.
4. Celery worker and beat services.
5. Nginx service serving static/media and proxying to Django.
6. `.env.example` committed and real `.env` ignored.
7. Migrations and collectstatic executed in containers.
8. App accessible from a browser or API client.
9. Logs checked with `docker compose logs`.
10. Optional HTTPS added after HTTP works.


## Summary

- Dockerized deployment uses multiple services connected by Compose.
- Nginx proxies to the `web` service and serves static/media volumes.
- Django/Gunicorn runs in the `web` container.
- PostgreSQL and Redis run as separate containers.
- Celery worker and beat are separate services using the same app image.
- Named volumes persist database, static, and media data.
- Logs and restart policies are essential for operations.
- Docker Compose is useful for single-server deployment; Kubernetes is the advanced orchestration path.
